# 뉴스 헤드라인 기반 낚시성(Clickbait) 기사 탐지

**Task**: Binary Classification (낚시성=1 / 정상=0)  
**Baseline**: TF-IDF + Logistic Regression  
**Advanced**: KcBERT 파인튜닝  
**평가지표**: Accuracy, F1-score

## 0. 환경 설정

In [ ]:
# 런타임 > 런타임 유형 변경 > T4 GPU 먼저 설정!
!pip install -q transformers==4.40.0 accelerate==0.28.0

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import torch
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트
!apt-get -qq install -y fonts-nanum
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('GPU:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. 데이터 업로드

In [ ]:
from google.colab import files

# 아래 실행하면 파일 선택 버튼이 뜸 → clickbait_data.csv 선택
uploaded = files.upload()

In [ ]:
df = pd.read_csv('clickbait_data.csv')
print(f'전체: {len(df):,}건')
print('클래스 분포:')
print(df['clickbait_class'].value_counts())

## 2. EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['clickbait_class'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color=['steelblue', 'tomato'], edgecolor='white')
axes[0].set_title('클래스 분포')
axes[0].set_xticklabels(['정상(0)', '낚시성(1)'], rotation=0)
axes[0].set_ylabel('개수')

df['title_len'] = df['title'].str.len()
for label, color, name in [(0,'steelblue','정상'), (1,'tomato','낚시성')]:
    df[df['clickbait_class']==label]['title_len'].plot(kind='kde', ax=axes[1], color=color, label=name)
axes[1].set_title('라벨별 제목 길이 분포')
axes[1].set_xlabel('글자 수')
axes[1].legend()

plt.tight_layout()
plt.savefig('eda.png', dpi=150, bbox_inches='tight')
plt.show()
print('정상 평균 길이:', round(df[df['clickbait_class']==0]['title_len'].mean(),1))
print('낚시성 평균 길이:', round(df[df['clickbait_class']==1]['title_len'].mean(),1))

## 3. 데이터 분리

In [ ]:
df_s = df.sample(n=100_000, random_state=42).reset_index(drop=True)
X, y = df_s['title'], df_s['clickbait_class']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_val,   X_test, y_val,   y_test  = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)
print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')

## 4. Baseline: TF-IDF + Logistic Regression

In [ ]:
tfidf = TfidfVectorizer(max_features=50_000, ngram_range=(1,2), analyzer='char_wb')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr.fit(X_train_tfidf, y_train)
bl_pred = lr.predict(X_test_tfidf)

bl_acc = accuracy_score(y_test, bl_pred)
bl_f1  = f1_score(y_test, bl_pred)
print(f'Baseline  Accuracy: {bl_acc:.4f} | F1: {bl_f1:.4f}')
print(classification_report(y_test, bl_pred, target_names=['정상','낚시성']))

cm = confusion_matrix(y_test, bl_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['정상','낚시성'], yticklabels=['정상','낚시성'])
plt.title('Baseline Confusion Matrix')
plt.ylabel('실제'); plt.xlabel('예측')
plt.tight_layout(); plt.savefig('baseline_cm.png', dpi=150, bbox_inches='tight'); plt.show()

## 5. Advanced: KcBERT 파인튜닝

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

MODEL_NAME = 'beomi/kcbert-base'
MAX_LEN    = 64
BATCH_SIZE = 32
EPOCHS     = 3
LR         = 2e-5

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('토크나이저 로드 완료')

In [ ]:
class ClickbaitDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts  = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = tokenizer(self.texts[idx], max_length=MAX_LEN,
                        padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(),
                'attention_mask': enc['attention_mask'].squeeze(),
                'label': torch.tensor(self.labels[idx], dtype=torch.long)}

train_loader = DataLoader(ClickbaitDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(ClickbaitDataset(X_val,   y_val),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(ClickbaitDataset(X_test,  y_test),  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f'배치 수 - Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer,
    num_warmup_steps=total_steps//10, num_training_steps=total_steps)
print('모델 로드 완료')

In [ ]:
def evaluate(loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch in loader:
            out = model(input_ids=batch['input_ids'].to(DEVICE),
                        attention_mask=batch['attention_mask'].to(DEVICE))
            preds.extend(out.logits.argmax(-1).cpu().numpy())
            trues.extend(batch['label'].numpy())
    return accuracy_score(trues, preds), f1_score(trues, preds), trues, preds

history = {'loss':[], 'val_acc':[], 'val_f1':[]}
best_f1 = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for step, batch in enumerate(train_loader):
        out = model(input_ids=batch['input_ids'].to(DEVICE),
                    attention_mask=batch['attention_mask'].to(DEVICE),
                    labels=batch['label'].to(DEVICE))
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step(); optimizer.zero_grad()
        total_loss += out.loss.item()
        if (step+1) % 200 == 0:
            print(f'  Epoch {epoch+1} Step {step+1}/{len(train_loader)} loss={out.loss.item():.4f}')

    val_acc, val_f1, _, _ = evaluate(val_loader)
    avg_loss = total_loss / len(train_loader)
    history['loss'].append(avg_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    print(f'[Epoch {epoch+1}] loss={avg_loss:.4f} | val_acc={val_acc:.4f} | val_f1={val_f1:.4f}')

    if val_f1 > best_f1:
        best_f1 = val_f1
        model.save_pretrained('/content/best_model')
        tokenizer.save_pretrained('/content/best_model')
        print('  → 모델 저장')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].plot(range(1,EPOCHS+1), history['loss'], marker='o')
axes[0].set_title('Train Loss'); axes[0].set_xlabel('Epoch')
axes[1].plot(range(1,EPOCHS+1), history['val_acc'], marker='o', label='Accuracy')
axes[1].plot(range(1,EPOCHS+1), history['val_f1'],  marker='s', label='F1')
axes[1].set_title('Validation 성능'); axes[1].legend()
plt.tight_layout(); plt.savefig('training_curve.png', dpi=150, bbox_inches='tight'); plt.show()

## 6. 최종 평가 및 모델 비교

In [ ]:
bert_acc, bert_f1, y_true, y_pred = evaluate(test_loader)

results = pd.DataFrame({
    '모델':      ['TF-IDF + LR (Baseline)', 'KcBERT (Advanced)'],
    'Accuracy': [round(bl_acc,4), round(bert_acc,4)],
    'F1-score': [round(bl_f1,4),  round(bert_f1,4)],
})
print('=== 모델 비교 (Test Set) ===')
print(results.to_string(index=False))
print()
print(classification_report(y_true, y_pred, target_names=['정상','낚시성']))

cm2 = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm2, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['정상','낚시성'], yticklabels=['정상','낚시성'])
plt.title('KcBERT Confusion Matrix')
plt.ylabel('실제'); plt.xlabel('예측')
plt.tight_layout(); plt.savefig('bert_cm.png', dpi=150, bbox_inches='tight'); plt.show()

## 7. 오분류 사례 질적 분석

In [ ]:
test_df = X_test.reset_index(drop=True).to_frame()
test_df['true'] = y_true
test_df['pred'] = y_pred

fp = test_df[(test_df['true']==0) & (test_df['pred']==1)]
fn = test_df[(test_df['true']==1) & (test_df['pred']==0)]

print(f'False Positive (정상→낚시성 오탐): {len(fp):,}건')
print(fp['title'].head(10).to_string())
print(f'\nFalse Negative (낚시성→정상 미탐): {len(fn):,}건')
print(fn['title'].head(10).to_string())

In [ ]:
# 결과 파일 다운로드
error_df = pd.concat([
    fp.head(20).assign(error_type='FP'),
    fn.head(20).assign(error_type='FN'),
])
error_df.to_csv('error_analysis.csv', index=False, encoding='utf-8-sig')
results.to_csv('model_comparison.csv', index=False, encoding='utf-8-sig')

files.download('eda.png')
files.download('baseline_cm.png')
files.download('training_curve.png')
files.download('bert_cm.png')
files.download('model_comparison.csv')
files.download('error_analysis.csv')
print('모든 결과 파일 다운로드 완료!')